# Fusión de Datasets: Clima y Cultivos

Con los datasets de cultivos y de clima ya limpios por separado, este notebook los une. El objetivo es simple de decir y un poco más laborioso de hacer bien: que cada registro de cultivo (una campaña, un cultivo, una provincia) quede con el clima que le correspondió ese año. Ese dataset fusionado es la base de todo el análisis que sigue.

## 1. Librerías y Carga de Datasets

### 1.1 Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

### 1.2 Carga de los Datasets Procesados

In [2]:
RUTA_PROYECTO = Path().resolve()
if RUTA_PROYECTO.name == "notebook":  # los notebooks se ejecutan desde la carpeta notebook/
    RUTA_PROYECTO = RUTA_PROYECTO.parent
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"

df_clima = pd.read_csv(RUTA_PROCESSED / "clima_procesado.csv")
df_cultivos = pd.read_csv(RUTA_PROCESSED / "cultivos_procesado.csv")
df_estaciones = pd.read_excel(RUTA_PROYECTO / "data" / "raw" / "estaciones_meteorologicas_2021-2026.xlsx")

In [3]:
print("Clima:", df_clima.shape)
print("Cultivos:", df_cultivos.shape)
print("Estaciones:", df_estaciones.shape)

Clima: (1171911, 15)
Cultivos: (138926, 11)
Estaciones: (92, 6)


### 1.3 Revisión de las Columnas de Texto del Dataset de Estaciones

Antes de usar `NOMBRE` y `PROVINCIA` como clave para cruzar datos, conviene asegurarse de que no traigan espacios sobrantes ni variantes de escritura — eso haría fallar un merge por texto de forma silenciosa (dos valores que a simple vista son iguales, pero no coinciden como string).

In [4]:
for col in ['NOMBRE', 'PROVINCIA']:
    con_espacios = (df_estaciones[col] != df_estaciones[col].str.strip()).sum()
    print(f"{col}: {con_espacios} valores con espacios al inicio/final")

print()
print("Cantidad de provincias únicas:", df_estaciones['PROVINCIA'].nunique())
print(sorted(df_estaciones['PROVINCIA'].unique()))

NOMBRE: 0 valores con espacios al inicio/final
PROVINCIA: 0 valores con espacios al inicio/final

Cantidad de provincias únicas: 24
['BUENOS AIRES', 'CATAMARCA', 'CHACO', 'CHUBUT', 'CIUDAD AUTÓNOMA DE BUENOS AIRES', 'CORRIENTES', 'CÓRDOBA', 'ENTRE RÍOS', 'FORMOSA', 'JUJUY', 'LA PAMPA', 'LA RIOJA', 'MENDOZA', 'MISIONES', 'NEUQUÉN', 'RÍO NEGRO', 'SALTA', 'SAN JUAN', 'SAN LUIS', 'SANTA CRUZ', 'SANTA FE', 'SANTIAGO DEL ESTERO', 'TIERRA DEL FUEGO', 'TUCUMÁN']


### 1.4 Verificación de Consistencia entre Provincias (Estaciones vs. Cultivos)

La fusión va a traer la provincia de cada estación meteorológica al dataset de cultivos. Para que eso funcione, los nombres de provincia tienen que escribirse igual en ambos datasets — verificamos acá si hay diferencias antes de intentar cruzarlos.

In [5]:
prov_estaciones = set(df_estaciones['PROVINCIA'].str.upper().unique())
prov_cultivos = set(df_cultivos['provincia'].dropna().str.upper().unique())

print("En estaciones pero no en cultivos:", prov_estaciones - prov_cultivos)
print("En cultivos pero no en estaciones:", prov_cultivos - prov_estaciones)
print()
print("Nulos en columna provincia (cultivos):", df_cultivos['provincia'].isna().sum())

En estaciones pero no en cultivos: {'CIUDAD AUTÓNOMA DE BUENOS AIRES'}
En cultivos pero no en estaciones: set()

Nulos en columna provincia (cultivos): 4


### 1.5 Estandarización de Provincia (Cultivos) para la Fusión

La verificación anterior mostró que la única diferencia real es de mayúsculas/minúsculas. Estandarizamos `provincia` en cultivos a mayúsculas, igual que ya está en estaciones.

In [6]:
df_cultivos['provincia'] = df_cultivos['provincia'].str.upper()

# Verificación: ya no debería haber diferencias (salvo CABA, que es esperado)
prov_estaciones = set(df_estaciones['PROVINCIA'].unique())
prov_cultivos = set(df_cultivos['provincia'].dropna().unique())
print("En estaciones pero no en cultivos:", prov_estaciones - prov_cultivos)
print("En cultivos pero no en estaciones:", prov_cultivos - prov_estaciones)

En estaciones pero no en cultivos: {'CIUDAD AUTÓNOMA DE BUENOS AIRES'}
En cultivos pero no en estaciones: set()


## 2. Incorporación de la Provincia al Dataset Climático

El dataset climático trae la estación meteorológica (`NRO_OMM`) que registró cada medición, pero no la provincia — y el dataset de cultivos está organizado por provincia, no por estación. Para poder cruzarlos, incorporamos la provincia de cada estación al clima.

**Decisión de granularidad geográfica:** el dataset de cultivos tiene nivel de detalle departamento (más fino que provincia), pero las estaciones meteorológicas del SMN no están geolocalizadas a nivel departamento — solo sabemos en qué provincia está cada una. Por eso, el cruce con clima va a quedar a nivel **provincia**, no departamento: es una simplificación real, no ideal, pero es el máximo nivel de detalle que los datos climáticos permiten. Esto significa que todos los departamentos de una misma provincia van a compartir el mismo valor climático anual — una limitación a tener en cuenta en el análisis exploratorio que viene después.

In [7]:
filas_antes = df_clima.shape[0]

df_clima = df_clima.merge(
    df_estaciones[['NRO_OMM', 'PROVINCIA']],
    on='NRO_OMM',
    how='left'
)

print("Filas antes del merge:", filas_antes)
print("Filas después del merge:", df_clima.shape[0])
print("Nulos en PROVINCIA:", df_clima['PROVINCIA'].isna().sum())

Filas antes del merge: 1171911
Filas después del merge: 1171911


Nulos en PROVINCIA: 0


In [8]:
df_clima['PROVINCIA'] = df_clima['PROVINCIA'].astype('category')
df_clima.dtypes

NRO_OMM         int64
NOMBRE            str
FECHA             str
TMAX          float64
TMIN          float64
TMEDIA        float64
PRECIP        float64
HELIOF        float64
VTO_MDIR      float64
VTO_MVEL      float64
VEL_MED       float64
PRE_EST       float64
HUM_REL       float64
NUB_TOTAL     float64
anio            int64
PROVINCIA    category
dtype: object

In [9]:
df_clima['FECHA'] = pd.to_datetime(df_clima['FECHA'])
df_clima['NOMBRE'] = df_clima['NOMBRE'].astype('category')

df_clima.dtypes

NRO_OMM               int64
NOMBRE             category
FECHA        datetime64[us]
TMAX                float64
TMIN                float64
TMEDIA              float64
PRECIP              float64
HELIOF              float64
VTO_MDIR            float64
VTO_MVEL            float64
VEL_MED             float64
PRE_EST             float64
HUM_REL             float64
NUB_TOTAL           float64
anio                  int64
PROVINCIA          category
dtype: object

### 2.1 Agregación Anual de Variables Climáticas por Provincia

El dataset de cultivos tiene una fila por campaña (año), así que agregamos el clima diario a esa misma granularidad: un valor por provincia y año. La mayoría de las variables se resume con el **promedio** anual (temperatura, humedad, nubosidad, etc.), salvo `PRECIP`, que se trata distinto: tiene sentido agronómico hablar del promedio de temperatura de un año, pero no del promedio de precipitación diaria; lo que importa es cuánto llovió en total.

**Corrección importante — cómo se calcula `PRECIP` provincial.** La precipitación anual de una provincia es el *promedio entre sus estaciones* del total anual de cada una, no la suma de todas las estaciones. En una primera versión de este notebook se sumaban todos los registros diarios de todas las estaciones de la provincia, y eso mide cuántas estaciones tiene la provincia más que cuánto llovió: Buenos Aires (21 estaciones) llegaba a ~16.000 mm anuales, un valor físicamente imposible. Con el cálculo correcto, la precipitación provincial queda en un rango plausible: entre 125 y 2.879 mm anuales, con una mediana cercana a 910 mm en las provincias y años que se cruzan con cultivos. Esta corrección cambió los resultados de correlación del notebook `04`.

Para que un año con huecos no subestime la lluvia, solo se usan las estaciones con dato en al menos 80 % de los días del año.

**Decisión de ventana temporal:** agregamos por año calendario (enero-diciembre), no por campaña agrícola (que para algunos cultivos cruza el fin de año). Es una simplificación consciente — la campaña real de un cultivo de invierno no coincide exactamente con el año calendario — pero evita la complejidad de definir una ventana de siembra-cosecha distinta para cada uno de los 8 cultivos. Queda documentada como limitación del análisis, no como un error.

In [10]:
cols_promedio = ['TMAX', 'TMIN', 'TMEDIA', 'HELIOF', 'VTO_MVEL', 'VEL_MED', 'PRE_EST', 'HUM_REL', 'NUB_TOTAL']

df_clima_promedios = df_clima.groupby(['PROVINCIA', 'anio'], observed=True)[cols_promedio].mean().reset_index()

# PRECIP: primero el total anual de cada estación (suma de sus días) y después el
# promedio entre las estaciones de la provincia. Se exige que la estación tenga dato
# en al menos el 80 % de los días del año (292 de 365), para que un año con huecos
# no subestime la lluvia anual.
DIAS_MINIMOS = 292

precip_estacion = (
    df_clima.groupby(['PROVINCIA', 'NRO_OMM', 'anio'], observed=True)['PRECIP']
    .agg(total_anual='sum', dias_con_dato='count')
    .reset_index()
)
print("Estación-año descartados por cobertura < 80 %:", (precip_estacion['dias_con_dato'] < DIAS_MINIMOS).sum(),
      "de", len(precip_estacion))
precip_estacion = precip_estacion[precip_estacion['dias_con_dato'] >= DIAS_MINIMOS]

df_clima_precip = (
    precip_estacion.groupby(['PROVINCIA', 'anio'], observed=True)['total_anual']
    .mean()
    .rename('PRECIP')
    .reset_index()
)

df_clima_anual = df_clima_promedios.merge(df_clima_precip, on=['PROVINCIA', 'anio'], how='left')

print(df_clima_anual.shape)
df_clima_anual.head()

Estación-año descartados por cobertura < 80 %: 120 de 3272
(864, 12)


,PROVINCIA,anio,TMAX,TMIN,TMEDIA,HELIOF,VTO_MVEL,VEL_MED,PRE_EST,HUM_REL,NUB_TOTAL,PRECIP
0,BUENOS AIRES,1991,21.129562,9.339838,15.634399,6.497843,NaN,NaN,1004.685463,NaN,NaN,974.227778
1,BUENOS AIRES,1992,20.794999,9.357175,15.441089,6.416648,NaN,NaN,1004.169134,NaN,NaN,1062.200000
2,BUENOS AIRES,1993,21.246910,9.384156,15.576153,6.700992,NaN,NaN,1004.071809,NaN,NaN,1172.511111
3,BUENOS AIRES,1994,21.529399,9.728234,15.783483,6.913054,NaN,NaN,1002.291212,NaN,NaN,906.305263
4,BUENOS AIRES,1995,21.643804,8.498248,15.298696,7.135069,NaN,NaN,1003.373549,NaN,NaN,748.005000


**Nota metodológica — Disponibilidad limitada de variables por modernización del SMN**

`VTO_MVEL`, `VEL_MED`, `HUM_REL` y `NUB_TOTAL` presentan 100% de nulos entre 1991 y 2020, y solo comienzan a reportarse de forma sistemática a partir de 2021, con la modernización de las estaciones automáticas del SMN. En la práctica, estas cuatro variables solo son analizables con un horizonte real de 6 años (2021-2026), muy por debajo del resto del dataset. Se conservan en la fusión (sin imputar ni descartar), documentando esta limitación para la fase exploratoria.

## 3. Fusión Final: Clima y Cultivos

Con el clima ya agregado por provincia y año, hacemos la fusión definitiva: cultivos como dataset ancla (se conservan todas sus filas) y un left join que le agrega las columnas climáticas correspondientes a su provincia y año. Las filas sin cobertura climática (antes de 1991, o sin provincia conocida) van a quedar con esas columnas en `NaN` — es el comportamiento esperado, no un error del merge.

In [11]:
filas_cultivos_antes = df_cultivos.shape[0]

df_fusion = df_cultivos.merge(
    df_clima_anual,
    left_on=['provincia', 'anio'],
    right_on=['PROVINCIA', 'anio'],
    how='left'
)

df_fusion = df_fusion.drop(columns=['PROVINCIA'])

print("Filas en cultivos antes del merge:", filas_cultivos_antes)
print("Filas en df_fusion después del merge:", df_fusion.shape[0])
print("Columnas:", df_fusion.shape[1])
print()
print("Filas con datos climáticos (TMAX no nulo):", df_fusion['TMAX'].notna().sum())
print("Filas sin datos climáticos (TMAX nulo):", df_fusion['TMAX'].isna().sum())

Filas en cultivos antes del merge: 138926
Filas en df_fusion después del merge: 138926
Columnas: 21

Filas con datos climáticos (TMAX no nulo): 47358
Filas sin datos climáticos (TMAX nulo): 91568


### 3.1 Verificación de Cobertura del Merge

Si el merge funcionó bien, toda fila que esté dentro del rango con cobertura climática (1991 en adelante) y tenga provincia conocida debería haber quedado con clima asignado. Verificamos que no haya ningún caso sin clima dentro de ese rango — si apareciera alguno, sería señal de un problema en el cruce, no una limitación esperada.

In [12]:
# Filas donde SÍ debería haber clima (año dentro del rango climático y con provincia conocida)
en_rango = df_fusion[(df_fusion['anio'] >= 1991) & (df_fusion['provincia'].notna())]
sin_clima_en_rango = en_rango['TMAX'].isna().sum()

print("Filas dentro del rango 1991-2026 con provincia conocida:", en_rango.shape[0])
print("De esas, cuántas quedaron SIN clima (debería ser 0):", sin_clima_en_rango)

Filas dentro del rango 1991-2026 con provincia conocida: 47358
De esas, cuántas quedaron SIN clima (debería ser 0): 0


## 4. Cierre de la Fase de Fusión

Con esto queda armado el dataset que sostiene el resto del proyecto: cada registro de cultivo (cultivo, provincia, año) con su clima correspondiente al lado. Es lo que permite, de acá en más, preguntarle a los datos si el clima realmente impactó el rendimiento — la pregunta que motivó todo el proyecto.

Quedan documentadas dos limitaciones reales, no errores: el cruce es a nivel **provincia**, no departamento (por el límite geográfico de las estaciones del SMN), y el clima se agregó por **año calendario**, no por campaña agrícola real de cada cultivo. Ninguna de las dos invalida el análisis, pero hay que tenerlas presentes al interpretar los resultados — sobre todo en cultivos cuya campaña cruza el fin de año.

El dataset fusionado se guarda en `data/processed`, listo para la exploración.

In [13]:
df_fusion.info()
df_fusion.head()

<class 'pandas.DataFrame'>
RangeIndex: 138926 entries, 0 to 138925
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   cultivo                  138926 non-null  str    
 1   anio                     138926 non-null  int64  
 2   campania                 138926 non-null  str    
 3   provincia                138922 non-null  object 
 4   provincia_id             138922 non-null  float64
 5   departamento             138052 non-null  str    
 6   departamento_id          138052 non-null  float64
 7   superficie_sembrada_ha   138926 non-null  float64
 8   superficie_cosechada_ha  132737 non-null  float64
 9   produccion_tm            132713 non-null  float64
 10  rendimiento_kgxha        132715 non-null  float64
 11  TMAX                     47358 non-null   float64
 12  TMIN                     47358 non-null   float64
 13  TMEDIA                   47358 non-null   float64
 14  HELIOF         

,cultivo,anio,campania,provincia,provincia_id,departamento,departamento_id,superficie_sembrada_ha,superficie_cosechada_ha,produccion_tm,...,TMAX,TMIN,TMEDIA,HELIOF,VTO_MVEL,VEL_MED,PRE_EST,HUM_REL,NUB_TOTAL,PRECIP
0,avena,1923,1923/1924,BUENOS AIRES,6.0,Adolfo Alsina,6007.0,8000.0,6700.0,6700.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,avena,1923,1923/1924,BUENOS AIRES,6.0,Adolfo Gonzales Chaves,6014.0,84000.0,78000.0,124800.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,avena,1923,1923/1924,BUENOS AIRES,6.0,Alberti,6021.0,4700.0,4200.0,4494.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,avena,1923,1923/1924,BUENOS AIRES,6.0,Ayacucho,6042.0,1780.0,1553.0,1508.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,avena,1923,1923/1924,BUENOS AIRES,6.0,Azul,6049.0,10000.0,8500.0,11390.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
RUTA_PROCESSED = RUTA_PROYECTO / "data" / "processed"
ruta_salida = RUTA_PROCESSED / "fusion_clima_cultivos.csv"
df_fusion.to_csv(ruta_salida, index=False, encoding='utf-8')